# Master Prompt Engineering and LangChain PromptTemplates

**Author:** Jack Pumpuni Frimpong-Manso ([GitHub: pumpuni07](https://github.com/pumpuni07))

Hands-on lab project completed as part of the **IBM RAG and Agentic AI Professional Certificate** (Coursera / IBM Skills Network). This notebook is my own reconstruction and extension of the lab *"Master Prompt Engineering and LangChain PromptTemplates"*, rebuilt independently for my portfolio.

## What this notebook covers

1. **Prompt engineering techniques** — basic prompting, zero-shot, one-shot, and few-shot in-context learning
2. **Advanced reasoning strategies** — chain-of-thought (CoT) prompting and self-consistency
3. **LangChain prompt templates** — `PromptTemplate`, `ChatPromptTemplate`, and `FewShotPromptTemplate`
4. **LCEL (LangChain Expression Language)** — composing chains with the pipe operator (`|`)
5. **Practical applications** — summarization, question answering, sentiment classification, and information extraction built as reusable LCEL chains

## Learning objectives

After working through this notebook you will be able to:

- Explain how prompt structure affects LLM output quality and reliability
- Apply zero-shot, one-shot, and few-shot prompting and know when each is appropriate
- Use chain-of-thought prompting to improve multi-step reasoning
- Implement self-consistency by sampling multiple reasoning paths and taking a majority vote
- Build reusable, parameterized prompts with LangChain prompt template classes
- Compose modular LLM pipelines with the modern LCEL `prompt | llm | parser` pattern

## 1. Setup

### 1.1 Install dependencies

The lab uses IBM watsonx.ai foundation models through the `langchain-ibm` integration. Pinned versions are listed in `requirements.txt`; the cell below installs the core packages if you are running this notebook in a fresh environment.

In [ ]:
# Uncomment to install in a fresh environment (e.g. Google Colab / Skills Network lab)
# %pip install --quiet langchain langchain-core langchain-ibm ibm-watsonx-ai

### 1.2 Initialize the LLM

This notebook uses an IBM Granite instruct model served on **watsonx.ai**.

- Inside the IBM Skills Network lab environment, `project_id="skills-network"` works without an API key.
- Outside that environment, set the environment variables `WATSONX_APIKEY` and `WATSONX_PROJECT_ID` from your own IBM Cloud account before running.
- Model catalogs change over time — if the model ID below is deprecated, check the [watsonx.ai supported foundation models documentation](https://www.ibm.com/products/watsonx-ai/foundation-models) for a current instruct model and substitute its ID.

Two model objects are created:

- `llm` — deterministic settings (greedy decoding) for reproducible outputs
- `llm_creative` — sampling enabled with a higher temperature, used later for self-consistency

In [ ]:
import os
from langchain_ibm import WatsonxLLM

WATSONX_URL = "https://us-south.ml.cloud.ibm.com"
MODEL_ID = "ibm/granite-3-3-8b-instruct"  # verify availability in the watsonx model catalog

# Project ID: "skills-network" works inside the Skills Network lab environment.
# Elsewhere, export WATSONX_PROJECT_ID and WATSONX_APIKEY for your own IBM Cloud project.
PROJECT_ID = os.environ.get("WATSONX_PROJECT_ID", "skills-network")

# Deterministic model — greedy decoding for reproducible outputs
llm = WatsonxLLM(
    model_id=MODEL_ID,
    url=WATSONX_URL,
    project_id=PROJECT_ID,
    params={
        "decoding_method": "greedy",
        "max_new_tokens": 256,
        "repetition_penalty": 1.05,
    },
)

# Sampling model — used for self-consistency (Section 3.2)
llm_creative = WatsonxLLM(
    model_id=MODEL_ID,
    url=WATSONX_URL,
    project_id=PROJECT_ID,
    params={
        "decoding_method": "sample",
        "max_new_tokens": 256,
        "temperature": 0.7,
        "top_p": 0.9,
    },
)

print("LLM initialized:", MODEL_ID)

A small helper keeps the output tidy throughout the notebook.

In [ ]:
def show(title: str, response: str) -> None:
    """Pretty-print a model response under a section title."""
    print(f"=== {title} ===")
    print(response.strip())
    print()

## 2. Prompt Engineering Techniques

A **prompt** is everything you send to the model: instructions, context, examples, and the input itself. The same model can produce dramatically different results depending on how the prompt is structured. This section walks through the core techniques in order of increasing sophistication.

### 2.1 Basic prompting

The simplest prompt is a bare instruction or question. This works for straightforward tasks, but gives the model no guidance about format, tone, or scope — so outputs can be verbose, unfocused, or inconsistently formatted.

In [ ]:
basic_prompt = "What is prompt engineering?"

response = llm.invoke(basic_prompt)
show("Basic prompt", response)

Adding **explicit constraints** (role, format, length) to the same question already improves control — this is the essence of prompt engineering.

In [ ]:
structured_prompt = (
    "You are a technical instructor. "
    "In exactly two sentences, define prompt engineering for a software developer audience."
)

response = llm.invoke(structured_prompt)
show("Structured prompt", response)

### 2.2 Zero-shot prompting

In **zero-shot** prompting, the model performs a task from the instruction alone, with **no examples**. This relies entirely on knowledge acquired during pre-training. It works well for common, well-defined tasks such as sentiment classification.

In [ ]:
zero_shot_prompt = """Classify the sentiment of the following review as Positive, Negative, or Neutral.

Review: The delivery was fast, but the packaging was damaged and one item was missing.
Sentiment:"""

response = llm.invoke(zero_shot_prompt)
show("Zero-shot classification", response)

### 2.3 One-shot prompting

**One-shot** prompting adds a single worked example before the actual input. The example demonstrates the expected format and level of detail — a form of **in-context learning**: the model adapts its behavior from the example without any weight updates.

In [ ]:
one_shot_prompt = """Convert the sentence into a formal business tone.

Example:
Informal: Hey, can you send me that file real quick?
Formal: Could you please send me the file at your earliest convenience?

Informal: We messed up the order, sorry about that.
Formal:"""

response = llm.invoke(one_shot_prompt)
show("One-shot tone conversion", response)

### 2.4 Few-shot prompting

**Few-shot** prompting provides several examples (typically 2–8). This is the most reliable in-context learning technique for tasks with non-obvious formats, custom label sets, or domain-specific conventions. The examples below teach the model a custom three-way label scheme for support-ticket routing.

In [ ]:
few_shot_prompt = """Route each customer message to one department: BILLING, TECHNICAL, or GENERAL.

Message: I was charged twice for my subscription this month.
Department: BILLING

Message: The app crashes every time I open the settings page.
Department: TECHNICAL

Message: What are your office opening hours during holidays?
Department: GENERAL

Message: My invoice shows the wrong VAT rate for Germany.
Department:"""

response = llm.invoke(few_shot_prompt)
show("Few-shot ticket routing", response)

**When to use which:**

| Technique | Examples given | Best for |
|---|---|---|
| Zero-shot | 0 | Common tasks the model already knows well |
| One-shot | 1 | Demonstrating a specific output format |
| Few-shot | 2–8 | Custom labels, domain conventions, higher reliability |

More examples generally improve consistency, at the cost of a longer prompt (more tokens, higher latency and cost).

## 3. Advanced Reasoning Strategies

### 3.1 Chain-of-thought (CoT) prompting

LLMs often fail multi-step problems when forced to answer immediately. **Chain-of-thought prompting** instructs the model to reason step by step before answering, which substantially improves performance on arithmetic, logic, and planning tasks.

First, the same word problem *without* CoT:

In [ ]:
problem = (
    "A warehouse receives 480 units on Monday. "
    "It ships out 35% of its stock on Tuesday, then receives 120 more units on Wednesday. "
    "How many units are in the warehouse at the end of Wednesday?"
)

direct_prompt = problem + "\nAnswer with a single number:"

response = llm.invoke(direct_prompt)
show("Direct answer (no CoT)", response)

Now the same problem *with* an explicit reasoning instruction:

In [ ]:
cot_prompt = problem + "\nLet's think step by step, then state the final answer on a new line as 'Final answer: <number>'."

response = llm.invoke(cot_prompt)
show("Chain-of-thought answer", response)

The correct reasoning: 480 units, ship 35% (168 units) leaving 312, then +120 = **432 units**. With CoT, the model exposes its intermediate steps, making errors visible and correctable.

### 3.2 Self-consistency

A single chain-of-thought can still go wrong. **Self-consistency** samples *multiple* independent reasoning paths (using a non-zero temperature) and takes a **majority vote** over the final answers. Correct reasoning paths tend to converge on the same answer, while errors scatter.

In [ ]:
import re
from collections import Counter

def extract_final_number(text: str):
    """Pull the last number that follows 'Final answer:' (fallback: last number in the text)."""
    match = re.findall(r"Final answer:\s*([\-\d.,]+)", text)
    candidates = match if match else re.findall(r"[\-\d.,]+", text)
    if not candidates:
        return None
    return candidates[-1].rstrip(".,").replace(",", "")

NUM_SAMPLES = 5
answers = []

for i in range(NUM_SAMPLES):
    sample = llm_creative.invoke(cot_prompt)
    answer = extract_final_number(sample)
    answers.append(answer)
    print(f"Sample {i + 1}: final answer = {answer}")

votes = Counter(a for a in answers if a is not None)
majority_answer, count = votes.most_common(1)[0]
print(f"\nSelf-consistency result: {majority_answer} ({count}/{NUM_SAMPLES} votes)")

Self-consistency trades compute (N model calls instead of 1) for reliability. In production it is typically reserved for high-stakes reasoning steps rather than every request.

## 4. LangChain Prompt Templates

Hard-coding prompt strings does not scale. LangChain's prompt template classes turn prompts into **reusable, parameterized components** with named input variables — the foundation for building composable applications.

### 4.1 `PromptTemplate` basics

`PromptTemplate.from_template()` infers input variables from `{placeholders}` in the template string.

In [ ]:
from langchain_core.prompts import PromptTemplate

summary_template = PromptTemplate.from_template(
    "Summarize the following text in {num_sentences} sentences for a {audience} audience:\n\n{text}"
)

print("Input variables:", summary_template.input_variables)

# Templates are formatted with .format() or .invoke() — no LLM call happens yet
formatted = summary_template.format(
    num_sentences=2,
    audience="non-technical",
    text="LangChain is an open source orchestration framework for building applications with large language models.",
)
print("\nFormatted prompt:\n" + formatted)

### 4.2 Composing chains with LCEL and the pipe operator

**LCEL (LangChain Expression Language)** composes components with the pipe operator (`|`), exactly like a Unix pipeline:

```
chain = prompt | llm | output_parser
```

- the **prompt template** formats the input dictionary into a prompt
- the **LLM** generates a completion
- the **output parser** (`StrOutputParser`) converts the raw model output into a clean string

The whole chain is executed with a single `.invoke()` call.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

summarize_chain = summary_template | llm | StrOutputParser()

article = (
    "The European Space Agency confirmed that its new Earth-observation satellite has entered "
    "its target orbit. The satellite carries a multispectral imaging instrument designed to "
    "monitor agricultural land use, forest cover, and coastal water quality. Data will be made "
    "freely available to researchers worldwide, supporting climate adaptation studies and "
    "precision-farming applications across Europe and Africa."
)

result = summarize_chain.invoke(
    {"num_sentences": 2, "audience": "non-technical", "text": article}
)
show("LCEL summarization chain", result)

### 4.3 `ChatPromptTemplate` — multi-role prompts

Chat models distinguish between **system** instructions (persistent behavior) and **human** messages (the actual request). `ChatPromptTemplate.from_messages()` builds structured multi-role prompts. This separation keeps behavioral instructions stable while the user input varies.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

qa_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise assistant. Answer ONLY from the provided context. "
     "If the context does not contain the answer, reply exactly: 'Not found in context.'"),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

qa_chain = qa_template | llm | StrOutputParser()

context = (
    "Sentinel-2 is a European wide-swath, high-resolution, multispectral imaging mission. "
    "It supports monitoring of vegetation, soil and water cover. The twin satellites fly in "
    "the same orbit, phased at 180 degrees, giving a revisit time of five days at the equator."
)

print(qa_chain.invoke({"context": context, "question": "What is the revisit time of Sentinel-2 at the equator?"}))
print()
print(qa_chain.invoke({"context": context, "question": "What is the launch mass of Sentinel-2?"}))

The second question is deliberately unanswerable from the context — a grounded chain should refuse rather than hallucinate. This grounding pattern is the conceptual core of **Retrieval-Augmented Generation (RAG)**.

### 4.4 `FewShotPromptTemplate` — structured in-context examples

Section 2.4 embedded few-shot examples as a raw string. `FewShotPromptTemplate` makes this systematic: examples live in a list of dictionaries, each rendered through its own `example_prompt`. Examples can now be stored, versioned, and selected dynamically.

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate

examples = [
    {"message": "I was charged twice for my subscription this month.", "department": "BILLING"},
    {"message": "The app crashes every time I open the settings page.", "department": "TECHNICAL"},
    {"message": "What are your office opening hours during holidays?", "department": "GENERAL"},
]

example_prompt = PromptTemplate.from_template(
    "Message: {message}\nDepartment: {department}"
)

routing_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Route each customer message to one department: BILLING, TECHNICAL, or GENERAL.",
    suffix="Message: {message}\nDepartment:",
    input_variables=["message"],
)

routing_chain = routing_template | llm | StrOutputParser()

test_messages = [
    "My invoice shows the wrong VAT rate for Germany.",
    "The export button produces an empty CSV file.",
    "Do you offer student discounts?",
]

for msg in test_messages:
    department = routing_chain.invoke({"message": msg}).strip()
    print(f"{msg}\n  -> {department}\n")

## 5. Practical Applications

Everything above combines into small, production-shaped applications. Each is a single LCEL chain built from a prompt template, the LLM, and a string parser.

### 5.1 Information extraction to structured output

In [ ]:
extraction_template = ChatPromptTemplate.from_messages([
    ("system",
     "Extract the requested fields from the text. "
     "Respond ONLY with valid JSON containing the keys: name, role, organization, location. "
     "Use null for any field not present. No markdown, no extra text."),
    ("human", "{text}"),
])

extraction_chain = extraction_template | llm | StrOutputParser()

bio = (
    "Dr. Amina Mensah joined the Leibniz Centre for Tropical Marine Research in Bremen "
    "last year as a senior data scientist, after five years leading analytics teams in Accra."
)

print(extraction_chain.invoke({"text": bio}))

### 5.2 Translation with tone control

In [ ]:
translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the user's text into {target_language} using a {tone} tone. Output only the translation."),
    ("human", "{text}"),
])

translation_chain = translation_template | llm | StrOutputParser()

result = translation_chain.invoke({
    "target_language": "German",
    "tone": "formal business",
    "text": "Thank you for your patience. We will resolve the issue by Friday and follow up with a full report.",
})
show("Formal German translation", result)

### 5.3 Chained reasoning: classify, then respond

LCEL chains compose: the output of one chain can feed the next. Here a support message is first **classified**, then a **department-appropriate reply** is drafted — a minimal two-step pipeline of the kind used in real support automation.

In [ ]:
reply_template = ChatPromptTemplate.from_messages([
    ("system",
     "You are a customer support agent in the {department} department. "
     "Write a brief (2-3 sentence), professional first reply to the customer message. "
     "Do not invent order numbers or policies."),
    ("human", "{message}"),
])

reply_chain = reply_template | llm | StrOutputParser()

customer_message = "The export button produces an empty CSV file."

# Step 1: route the message using the few-shot chain from Section 4.4
department = routing_chain.invoke({"message": customer_message}).strip().split()[0]
print("Routed to:", department)

# Step 2: draft a reply conditioned on the routing result
reply = reply_chain.invoke({"department": department, "message": customer_message})
show("Drafted reply", reply)

## 6. Exercises

Try these on your own before checking the sample solutions.

1. **Few-shot NER:** Build a `FewShotPromptTemplate` that extracts city names from sentences, using three examples. Test it on a sentence containing two cities.
2. **CoT + self-consistency:** Apply the self-consistency loop from Section 3.2 to this problem: *"A tank holds 900 liters. Pump A drains 15 liters/min, pump B fills 9 liters/min. Both run simultaneously. After how many minutes is the tank empty?"* (Correct answer: 150 minutes.)
3. **Grounded Q&A refusal:** Modify the Q&A chain in Section 4.3 so the refusal message is a template variable instead of a hard-coded string.

In [ ]:
# Exercise 1 — sample solution
ner_examples = [
    {"sentence": "She flew from Accra to Bremen last spring.", "cities": "Accra, Bremen"},
    {"sentence": "The conference alternates between Nairobi and Cape Town.", "cities": "Nairobi, Cape Town"},
    {"sentence": "He has never left Kumasi.", "cities": "Kumasi"},
]

ner_example_prompt = PromptTemplate.from_template("Sentence: {sentence}\nCities: {cities}")

ner_template = FewShotPromptTemplate(
    examples=ner_examples,
    example_prompt=ner_example_prompt,
    prefix="Extract all city names from the sentence as a comma-separated list.",
    suffix="Sentence: {sentence}\nCities:",
    input_variables=["sentence"],
)

ner_chain = ner_template | llm | StrOutputParser()
print(ner_chain.invoke({"sentence": "The night train runs from Hamburg to Vienna twice a week."}))

In [ ]:
# Exercise 2 — sample solution
tank_problem = (
    "A tank holds 900 liters. Pump A drains 15 liters per minute, pump B fills 9 liters per minute. "
    "Both run simultaneously. After how many minutes is the tank empty?"
    "\nLet's think step by step, then state the final answer on a new line as 'Final answer: <number>'."
)

tank_answers = []
for i in range(5):
    sample = llm_creative.invoke(tank_problem)
    tank_answers.append(extract_final_number(sample))
    print(f"Sample {i + 1}: {tank_answers[-1]}")

tank_votes = Counter(a for a in tank_answers if a is not None)
print("\nMajority answer:", tank_votes.most_common(1)[0])

In [ ]:
# Exercise 3 — sample solution
qa_template_v2 = ChatPromptTemplate.from_messages([
    ("system",
     "You are a precise assistant. Answer ONLY from the provided context. "
     "If the context does not contain the answer, reply exactly: '{refusal_message}'"),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

qa_chain_v2 = qa_template_v2 | llm | StrOutputParser()

print(qa_chain_v2.invoke({
    "context": context,
    "question": "What is the launch mass of Sentinel-2?",
    "refusal_message": "I cannot answer this from the given context.",
}))

## 7. Conclusion

This lab progressed from basic prompting through zero-shot, one-shot, and few-shot in-context learning, to chain-of-thought reasoning and self-consistency, and finally to LangChain's prompt template classes composed into applications with the modern LCEL pipe-operator pattern.

Key takeaways:

- **Prompt structure is an engineering surface**: constraints, roles, and examples directly control output quality.
- **In-context learning** (one-shot / few-shot) reliably teaches format and custom label schemes without any fine-tuning.
- **Chain-of-thought** exposes intermediate reasoning; **self-consistency** hardens it via majority voting.
- **`PromptTemplate` / `ChatPromptTemplate` / `FewShotPromptTemplate`** turn prompts into versioned, reusable components.
- **LCEL (`prompt | llm | parser`)** makes chains modular and composable — the foundation for RAG pipelines and agentic systems built later in this Professional Certificate.

## Acknowledgments

This notebook is an independent reconstruction of the IBM Skills Network lab *"Master Prompt Engineering and LangChain PromptTemplates"* from the IBM RAG and Agentic AI Professional Certificate. Original lab authors: **Hailey Quach**, **Kang Wang**, and **Faranak Heidari** (Data Scientists at IBM). All code in this notebook was written and executed by me for portfolio purposes.